#### In this notebook, the goal is to:
1. Combine the df_genre (from ticketmaster API) and df_skiddle_genre (from skiddle API).
2. Remove the duplicate fields - on name and date. I cant do it on the ids since they are not pointing to the same thing
3. df_genre (from ticketmaster API) renamed as 'df_tm'.
4. df_skiddle_genre (from skiddle API) renamed as 'df_sk'

In [3]:
from dotenv import load_dotenv
import os
import requests
from pprint import pprint
import pandas as pd
import missingno as mno
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
import time
import missingno as mno

In [8]:
df_tm = pd.read_csv("df_genre.csv")
df_sk = pd.read_csv("df_skiddle_genre.csv")

In [13]:
df_tm.head()

,id,name,dates.start.localDate,genre_name,subGenre_name,city_name,attraction_count,attraction_genres
0,LvZ18QxAj1bZeL8vGSGnc,EVERYWHERE AT ONCE: Dewin,2026-06-28,Alternative,Adult Alternative Pop/Rock,"Narberth, Pembrokeshire",2.0,"['Alternative', 'Other']"
1,LvZ18QO6E5_0DZYvqZ7DL,King Tuts Summer Nights - Golden Ticket,2026-07-09,Rock,Pop,Glasgow,1.0,['Rock']
2,16djZbzkpG7X9uN,Friday Day - Wireless 2026,2026-07-10,Undefined,Undefined,London,2.0,"['Undefined', 'Hip-Hop/Rap']"
3,1AMZkGyGkdEauCg,Amex Presents BST Hyde Park - Pitbull - Ultima...,2026-07-10,Undefined,Undefined,London,3.0,"['Undefined', 'Pop']"
4,1AMZkGyGkeJTygI,American Express Presents BST Hyde Park - Pitbull,2026-07-10,Undefined,Undefined,London,6.0,"['Undefined', 'Pop', 'World', 'Hip-Hop/Rap']"


In [14]:
print(df_tm.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2046 entries, 0 to 2045
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     2046 non-null   object 
 1   name                   2046 non-null   object 
 2   dates.start.localDate  2046 non-null   object 
 3   genre_name             2046 non-null   object 
 4   subGenre_name          2046 non-null   object 
 5   city_name              2046 non-null   object 
 6   attraction_count       1969 non-null   float64
 7   attraction_genres      1969 non-null   object 
dtypes: float64(1), object(7)
memory usage: 128.0+ KB
None


In [9]:
df_sk.head()

,id,eventname,startdate,enddate,venue.town,venue.region,cancelled,genre_count,genres,artist_count
0,41585614,Forbidden Forest 2026,2026-06-04T12:00:00+00:00,2026-06-07T23:00:00+00:00,Grantham,Nottingham,0,4.0,"['House', 'Drum and Bass', 'Techno', 'Tech Hou...",50.0
1,41643401,Bulletproof Festival 2026,2026-06-04T17:30:00+00:00,2026-06-06T22:30:00+00:00,London,London,0,NaN,NaN,26.0
2,41392590,Symphonic Ibiza - At The Beach,2026-06-05T15:00:00+00:00,2026-06-05T22:30:00+00:00,Weston Super-Mare,Bristol,0,NaN,NaN,2.0
3,41997035,AnExperience Festival,2026-06-05T14:00:00+00:00,2026-06-08T00:00:00+00:00,Huntingdon,Cambridgeshire,0,5.0,"['Dancehall', 'Ska', 'Salsa', 'World Music', '...",16.0
4,40660839,Fields of Éire- Irish Music Festival Liverpool...,2026-06-05T16:30:00+00:00,2026-06-06T23:00:00+00:00,Liverpool,Merseyside,0,3.0,"['Acoustic', 'Folk', 'Country/Americana']",NaN


In [15]:
print(df_sk.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1056 entries, 0 to 1055
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            1056 non-null   int64  
 1   eventname     1056 non-null   object 
 2   startdate     1056 non-null   object 
 3   enddate       1056 non-null   object 
 4   venue.town    1055 non-null   object 
 5   venue.region  1053 non-null   object 
 6   cancelled     1056 non-null   int64  
 7   genre_count   829 non-null    float64
 8   genres        829 non-null    object 
 9   artist_count  620 non-null    float64
dtypes: float64(2), int64(2), object(6)
memory usage: 82.6+ KB
None


##### I found 'Undefined' in genre & subgenre of df_tm. Only 15.5% of Ticketmaster events have Undefined genre, and they all overlap with subgenre so will keep this.

In [17]:
print(type(df_tm['attraction_genres'].iloc[0]))
print(df_tm['attraction_genres'].iloc[0])

<class 'str'>
['Alternative', 'Other']


In [18]:
import ast

def safe_parse(val):
    if pd.isna(val):
        return []
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return []

df_tm['attraction_genres'] = df_tm['attraction_genres'].apply(safe_parse)
df_sk['genres'] = df_sk['genres'].apply(safe_parse)

In [20]:
all_tm_genres = set()
for genre_list in df_tm['attraction_genres'].dropna():
    if isinstance(genre_list, list):
        all_tm_genres.update(genre_list)

print(f"Distinct genres in attraction_genres: {len(all_tm_genres)}")
pprint(sorted(all_tm_genres))

Distinct genres in attraction_genres: 33
['Alternative',
 'Blues',
 'Casino/Gaming',
 "Children's Music",
 "Children's Theatre",
 'Classical',
 'Comedy',
 'Country',
 'Dance/Electronic',
 'Drama',
 'Family',
 'Folk',
 'Food & Drink',
 'Foreign',
 'Hip-Hop/Rap',
 'Jazz',
 'Latin',
 'Metal',
 'Miscellaneous',
 'Miscellaneous Theatre',
 'New Age',
 'Other',
 'Performance Art',
 'Pop',
 'R&B',
 'Reggae',
 'Religious',
 'Rock',
 'Rugby',
 'Theatre',
 'Undefined',
 'Variety',
 'World']


In [22]:
all_sk_genres = set()
for genre_list in df_sk['genres'].dropna():
    if isinstance(genre_list, list):
        all_sk_genres.update(genre_list)

print(f"Distinct genres in genres: {len(all_sk_genres)}")
pprint(sorted(all_sk_genres))

Distinct genres in genres: 106
['1960s',
 '1970s',
 '1980s',
 '1990s',
 '2000s',
 '2010s',
 'Acid House',
 'Acoustic',
 'African',
 'Afrobeat',
 'Alternative',
 'Alternative Pop',
 'Amapiano',
 'Bass Music',
 'Bassline',
 'Big Band',
 'Big Beat',
 'Blues',
 'Bollywood',
 'Bounce',
 'Breaks',
 'Brit Pop',
 'Burlesque',
 'Cheesy Dance',
 'Choral',
 'Classical',
 'Club Classics',
 'Country/Americana',
 'Covers Band/Tribute Act',
 'Dancehall',
 'Death Rock',
 'Deep House',
 'Disco',
 'Disco House',
 'Drum and Bass',
 'Dub',
 'Dubstep',
 'EDM',
 'Electro',
 'Electro House',
 'Electro Swing',
 'Electronic',
 'Emo',
 'Experimental',
 'Folk',
 'Funk',
 'Funky House',
 'Grime',
 'Grunge',
 'Hard Dance',
 'Hard House',
 'Hard Rock',
 'Hard Trance',
 'Hardcore/Hardstyle',
 'Hip Hop',
 'House',
 'Indie',
 'Indie Pop',
 'J-pop',
 'Jazz',
 'Jungle',
 'K-pop',
 'LGBTQ+',
 'Latin',
 'Metal',
 'Minimal',
 'Minimal Techno',
 'New Wave',
 'Northern Soul',
 'Nu Disco',
 'Nu Indie',
 'Nu Metal',
 'Nu Rave'

In [23]:
print((df_tm['genre_name'] == 'Undefined').sum())
print((df_tm['subGenre_name'] == 'Undefined').sum())
print('Undefined' in all_tm_genres)  # from your attraction_genres flattening

317
317
True


In [24]:
print(f"{317/len(df_tm):.1%} of Ticketmaster events have Undefined genre")

15.5% of Ticketmaster events have Undefined genre


In [26]:
df_tm_known_genre = df_tm[df_tm['genre_name'] != 'Undefined'].copy()
print(f"Events with known genre: {len(df_tm_known_genre)} out of {len(df_tm)}")

Events with known genre: 1729 out of 2046


#### Dealing with duplicates:

##### Step 1: Normalize both date columns to real datetime, then to just the date (no time)

In [27]:
df_tm['date_clean'] = pd.to_datetime(df_tm['dates.start.localDate']).dt.date
df_sk['date_clean'] = pd.to_datetime(df_sk['startdate']).dt.date

print(df_tm['date_clean'].dtype)
print(df_sk['date_clean'].dtype)

object
object


In [30]:
print(df_tm['date_clean'].head())
print(type(df_tm['date_clean'].iloc[0]))

0    2026-06-28
1    2026-07-09
2    2026-07-10
3    2026-07-10
4    2026-07-10
Name: date_clean, dtype: object
<class 'datetime.date'>


##### Step 2: Normalize event names

In [40]:
def normalize_name(name):
    if pd.isna(name):
        return ""
    return name.lower().strip()

df_tm['name_clean'] = df_tm['name'].apply(normalize_name)
df_sk['name_clean'] = df_sk['eventname'].apply(normalize_name)

In [41]:
print(df_tm['name_clean'].head())
print(df_sk['name_clean'].head())

0                            everywhere at once: dewin
1              king tuts summer nights - golden ticket
2                           friday day - wireless 2026
3    amex presents bst hyde park - pitbull - ultima...
4    american express presents bst hyde park - pitbull
Name: name_clean, dtype: object
0                                forbidden forest 2026
1                            bulletproof festival 2026
2                       symphonic ibiza - at the beach
3                                anexperience festival
4    fields of éire- irish music festival liverpool...
Name: name_clean, dtype: object


In [42]:
print(df_tm['name'].head())
print(df_sk['eventname'].head())

0                            EVERYWHERE AT ONCE: Dewin
1              King Tuts Summer Nights - Golden Ticket
2                           Friday Day - Wireless 2026
3    Amex Presents BST Hyde Park - Pitbull - Ultima...
4    American Express Presents BST Hyde Park - Pitbull
Name: name, dtype: object
0                                Forbidden Forest 2026
1                            Bulletproof Festival 2026
2                       Symphonic Ibiza - At The Beach
3                                AnExperience Festival
4    Fields of Éire- Irish Music Festival Liverpool...
Name: eventname, dtype: object


In [58]:
pprint(df_tm['name'].sample(10).tolist())

['SUPERTONIC Fri 24 July 2026 The Jam House Birmingham',
 'Latitude Luxury 2026 - Bedouin Tent for 2 or 4',
 'Isabelle Mettle',
 'ACUA',
 'Solya',
 'Self Esteem - Upgrade (does not include event ticket)',
 'Day Fever',
 'Alewya',
 'Old School Indie LONDON - Over 30s Daytime Party, Sat 15th August, 3pm-7pm',
 'HELLBENT FOREVER - The Ultimate Judas Priest Tribute']


In [51]:
pprint(df_sk['eventname'].sample(10).tolist())

['Aitch Live at Abbey Park Leicester 2026',
 'London Mahotsav',
 'Hits On The Pitch 2026',
 'Lytham Festival',
 'The Great Fete',
 'Garage Republic Festival 2026 | Motorpoint Arena',
 'Summer Fest at the Beach Parking',
 'Flavours of Yorkshire',
 'Slug Fest 2026',
 'Festwich Tribute Festival']


##### Step 3: Trying exact match

In [73]:
exact_matches = df_tm.merge(
    df_sk, on=['name_clean', 'date_clean'],
    how='inner', suffixes=('_tm', '_sk')
)
print(f"Exact matches found: {len(exact_matches)}")
exact_matches[['name', 'eventname', 'date_clean']].head(10)

Exact matches found: 17


,name,eventname,date_clean
0,90s & 00s Nation Summer Fest,90s & 00s Nation Summer Fest,2026-07-10
1,In The Park Newcastle Presents Paul Weller,In the Park Newcastle presents Paul Weller,2026-07-10
2,Boogietown 2026,Boogietown 2026,2026-07-11
3,Electric Heart Surrey,Electric Heart Surrey,2026-07-12
4,In the Park Newcastle presents Wolf Alice,In the Park Newcastle presents Wolf Alice,2026-07-12
5,Burna Boy LIVE at The Milton Keynes National Bowl,Burna Boy LIVE at The Milton Keynes National Bowl,2026-07-31
6,The Kooks Inside In / Inside Out 20 year anniv...,The Kooks Inside In / Inside Out 20 year anniv...,2026-07-31
7,Dom Dolla on the Thames,Dom Dolla on the Thames,2026-08-01
8,Peggy Gou on the Thames,Peggy Gou on the Thames,2026-08-02
9,Electric Paradise,Electric Paradise,2026-08-08


##### Step 4: Removing the exact matches

In [76]:
tm_ids_to_drop = exact_matches['id_tm']

df_tm_deduped = df_tm[~df_tm['id'].isin(tm_ids_to_drop)].copy()

print(f"Ticketmaster before: {len(df_tm)}")
print(f"Ticketmaster after removing exact-match duplicates: {len(df_tm_deduped)}")

Ticketmaster before: 2046
Ticketmaster after removing exact-match duplicates: 2030


##### Step 4: Genre matching

In [81]:
all_tm_genres = set()
for g_list in df_tm['attraction_genres'].dropna():
    if isinstance(g_list, list):
        all_tm_genres.update(g_list)
all_tm_genres.update(df_tm['genre_name'].dropna().unique())
all_tm_genres.update(df_tm['subGenre_name'].dropna().unique())

all_sk_genres = set()
for g_list in df_sk['genres'].dropna():
    if isinstance(g_list, list):
        all_sk_genres.update(g_list)

print("TICKETMASTER genres:", sorted(all_tm_genres))
print("\nSKIDDLE genres:", sorted(all_sk_genres))

TICKETMASTER genres: ['Acoustic Blues', 'Adult Alternative Pop/Rock', 'Adult Contemporary', 'African', 'Afro-Beat', 'Alternative', 'Alternative Folk', 'Alternative Rock', 'Amapiano', 'Ambient', 'Americana', 'Big Band', 'Blues', 'Blues-Rock', 'Bollywood', 'British Rap', 'Casino/Gaming', 'Celtic Folk', 'Celtic/ British Isles', "Children's Music", "Children's Theatre", 'Classic Country', 'Classic Rock', 'Classical', 'Classical/Vocal', 'Club Dance', 'Comedy', 'Community/Civic', 'Contemporary Country', 'Country', 'Country Pop', 'Cuban', 'Cuban Jazz', 'Dance Pop', 'Dance/Electronic', 'Death Metal/Black Metal', 'Disco', 'Drama', 'Dream Pop', 'Electro Pop', 'Electro-Jazz', 'Electronic', 'Family', 'Folk', 'Food & Drink', 'Foreign', 'Free Jazz', 'Funk', 'Fusion', 'Garage Rock', 'German Rock', 'Glam', 'Gospel', 'Goth', 'Goth Metal', 'Grunge', 'Hair Metal', 'Hard Rock', 'Heavy Metal', 'Hip-Hop/Rap', 'House', 'Indie Folk', 'Indie Pop', 'Indie Rock', 'Jazz', 'Jazz Blues', 'Jazz Funk', 'Latin', 'Lati

In [82]:
genre_bucket_map = {
    # Rock / Indie / Alternative
    'Alternative': 'Rock/Indie', 'Alternative Rock': 'Rock/Indie', 'Alternative Pop': 'Rock/Indie',
    'Adult Alternative Pop/Rock': 'Rock/Indie', 'Blues-Rock': 'Rock/Indie', 'Classic Rock': 'Rock/Indie',
    'Garage Rock': 'Rock/Indie', 'German Rock': 'Rock/Indie', 'Glam': 'Rock/Indie', 'Grunge': 'Rock/Indie',
    'Hard Rock': 'Rock/Indie', 'Indie Pop': 'Rock/Indie', 'Indie Rock': 'Rock/Indie', 'Indie': 'Rock/Indie',
    'Nu Indie': 'Rock/Indie', 'Pop Rock': 'Rock/Indie', 'Progressive Rock': 'Rock/Indie',
    'Psychedelic': 'Rock/Indie', 'Rock': 'Rock/Indie', 'Rock & Roll': 'Rock/Indie',
    'Rock & Roll / 1950s': 'Rock/Indie', 'Brit Pop': 'Rock/Indie', 'New Wave': 'Rock/Indie',
    'Post Rock': 'Rock/Indie', 'Shoegaze': 'Rock/Indie',

    # Pop
    'Pop': 'Pop', 'Adult Contemporary': 'Pop', 'Dream Pop': 'Pop', 'J-pop': 'Pop', 'K-pop': 'Pop',
    'Synth Pop': 'Pop',

    # Hip-Hop / Urban / R&B
    'Hip-Hop/Rap': 'Hip-Hop/Urban', 'Hip Hop': 'Hip-Hop/Urban', 'British Rap': 'Hip-Hop/Urban',
    'R&B': 'Hip-Hop/Urban', 'Rap': 'Hip-Hop/Urban', 'Rap-Rock': 'Hip-Hop/Urban', 'Latin Rap': 'Hip-Hop/Urban',
    'Trap': 'Hip-Hop/Urban', 'Urban': 'Hip-Hop/Urban', 'Grime': 'Hip-Hop/Urban',

    # Electronic / Dance
    'Amapiano': 'Electronic/Dance', 'Ambient': 'Electronic/Dance', 'Club Dance': 'Electronic/Dance',
    'Dance Pop': 'Electronic/Dance', 'Dance/Electronic': 'Electronic/Dance', 'Disco': 'Electronic/Dance',
    'Electro Pop': 'Electronic/Dance', 'Electro-Jazz': 'Electronic/Dance', 'Electronic': 'Electronic/Dance',
    'House': 'Electronic/Dance', 'New Age': 'Electronic/Dance', 'Techno': 'Electronic/Dance',
    'Acid House': 'Electronic/Dance', 'Bass Music': 'Electronic/Dance', 'Bassline': 'Electronic/Dance',
    'Big Beat': 'Electronic/Dance', 'Bounce': 'Electronic/Dance', 'Breaks': 'Electronic/Dance',
    'Cheesy Dance': 'Electronic/Dance', 'Club Classics': 'Electronic/Dance', 'Deep House': 'Electronic/Dance',
    'Disco House': 'Electronic/Dance', 'Drum and Bass': 'Electronic/Dance', 'Dubstep': 'Electronic/Dance',
    'EDM': 'Electronic/Dance', 'Electro': 'Electronic/Dance', 'Electro House': 'Electronic/Dance',
    'Electro Swing': 'Electronic/Dance', 'Experimental': 'Electronic/Dance', 'Funky House': 'Electronic/Dance',
    'Hard Dance': 'Electronic/Dance', 'Hard House': 'Electronic/Dance', 'Hard Trance': 'Electronic/Dance',
    'Hardcore/Hardstyle': 'Electronic/Dance', 'Jungle': 'Electronic/Dance', 'Minimal': 'Electronic/Dance',
    'Minimal Techno': 'Electronic/Dance', 'Nu Disco': 'Electronic/Dance', 'Nu Rave': 'Electronic/Dance',
    'Old Skool': 'Electronic/Dance', 'Prog House': 'Electronic/Dance', 'Psy/GoaTrance': 'Electronic/Dance',
    'Retro House': 'Electronic/Dance', 'Soulful House': 'Electronic/Dance', 'Tech House': 'Electronic/Dance',
    'Trance': 'Electronic/Dance', 'Tribal House': 'Electronic/Dance', 'UK Garage': 'Electronic/Dance',

    # Folk / Country / Americana
    'Alternative Folk': 'Folk/Country', 'Americana': 'Folk/Country', 'Celtic Folk': 'Folk/Country',
    'Celtic/ British Isles': 'Folk/Country', 'Classic Country': 'Folk/Country', 'Contemporary Country': 'Folk/Country',
    'Country': 'Folk/Country', 'Country Pop': 'Folk/Country', 'Folk': 'Folk/Country',
    'Indie Folk': 'Folk/Country', 'Scottish Folk': 'Folk/Country', 'Singer-Songwriter': 'Folk/Country',
    'Traditional Scottish Folk': 'Folk/Country', 'Acoustic': 'Folk/Country', 'Country/Americana': 'Folk/Country',

    # Jazz / Blues / Soul
    'Acoustic Blues': 'Jazz/Blues/Soul', 'Big Band': 'Jazz/Blues/Soul', 'Blues': 'Jazz/Blues/Soul',
    'Cuban Jazz': 'Jazz/Blues/Soul', 'Free Jazz': 'Jazz/Blues/Soul', 'Funk': 'Jazz/Blues/Soul',
    'Fusion': 'Jazz/Blues/Soul', 'Gospel': 'Jazz/Blues/Soul', 'Jazz': 'Jazz/Blues/Soul',
    'Jazz Blues': 'Jazz/Blues/Soul', 'Jazz Funk': 'Jazz/Blues/Soul', 'Motown': 'Jazz/Blues/Soul',
    'Neo-Soul': 'Jazz/Blues/Soul', 'Northern Soul': 'Jazz/Blues/Soul', 'Psychedelic Soul': 'Jazz/Blues/Soul',
    'Soul': 'Jazz/Blues/Soul', 'Soul Jazz': 'Jazz/Blues/Soul', 'Swing': 'Jazz/Blues/Soul',

    # Metal / Punk
    'Death Metal/Black Metal': 'Metal/Punk', 'Goth': 'Metal/Punk', 'Goth Metal': 'Metal/Punk',
    'Hair Metal': 'Metal/Punk', 'Heavy Metal': 'Metal/Punk', 'Metal': 'Metal/Punk', 'Metalcore': 'Metal/Punk',
    'Nu-Metal': 'Metal/Punk', 'Nu Metal': 'Metal/Punk', 'Post-Punk': 'Metal/Punk', 'Power Metal': 'Metal/Punk',
    'Punk': 'Metal/Punk', 'Thrash & Speed': 'Metal/Punk', 'Death Rock': 'Metal/Punk', 'Emo': 'Metal/Punk',
    'Pop Punk': 'Metal/Punk',

    # Classical / Orchestral
    'Classical': 'Classical/Orchestral', 'Classical/Vocal': 'Classical/Orchestral',
    'Choral': 'Classical/Orchestral', 'Orchestral': 'Classical/Orchestral',

    # World / Latin / Reggae
    'African': 'World/Latin/Reggae', 'Afro-Beat': 'World/Latin/Reggae', 'Afrobeat': 'World/Latin/Reggae',
    'Bollywood': 'World/Latin/Reggae', 'Cuban': 'World/Latin/Reggae', 'Latin': 'World/Latin/Reggae',
    'Pakistan': 'World/Latin/Reggae', 'Political Reggae': 'World/Latin/Reggae', 'Reggae': 'World/Latin/Reggae',
    'Rock Steady': 'World/Latin/Reggae', 'Roots Reggae': 'World/Latin/Reggae', 'Uganda': 'World/Latin/Reggae',
    'World': 'World/Latin/Reggae', 'World Dance': 'World/Latin/Reggae', 'World Music': 'World/Latin/Reggae',
    'Dancehall': 'World/Latin/Reggae', 'Dub': 'World/Latin/Reggae', 'Salsa': 'World/Latin/Reggae',
    'Ska': 'World/Latin/Reggae',

    # Retro / decade-themed (not really a genre — a framing device)
    '1960s': 'Retro/Decades', '1970s': 'Retro/Decades', '1980s': 'Retro/Decades', '1990s': 'Retro/Decades',
    '2000s': 'Retro/Decades', '2010s': 'Retro/Decades', 'Retro and Throwbacks': 'Retro/Decades',

    # Non-music / mixed event types
    'Casino/Gaming': 'Non-Music/Other', "Children's Music": 'Non-Music/Other',
    "Children's Theatre": 'Non-Music/Other', 'Comedy': 'Non-Music/Other', 'Community/Civic': 'Non-Music/Other',
    'Drama': 'Non-Music/Other', 'Family': 'Non-Music/Other', 'Food & Drink': 'Non-Music/Other',
    'Miscellaneous Theatre': 'Non-Music/Other', 'Performance Art': 'Non-Music/Other',
    'Religious': 'Non-Music/Other', 'Rugby': 'Non-Music/Other', 'Rugby Union': 'Non-Music/Other',
    'Theatre': 'Non-Music/Other', 'Variety': 'Non-Music/Other', 'Burlesque': 'Non-Music/Other',
    'Spoken Word': 'Non-Music/Other', 'LGBTQ+': 'Non-Music/Other',

    # Unclassified
    'Undefined': 'Unclassified', 'Other': 'Unclassified', 'Miscellaneous': 'Unclassified',
    'Foreign': 'Unclassified', 'Covers Band/Tribute Act': 'Unclassified', 'Themed': 'Unclassified',
}

def map_to_bucket(genre_name):
    return genre_bucket_map.get(genre_name, 'Other/Unmapped')

In [87]:
df_tm['genres_bucketed'] = df_tm['genre_bucket'].apply(lambda x: [x])
df_sk['genres_bucketed'] = df_sk['genre_buckets']

In [83]:
# Ticketmaster — apply to genre_name, subGenre_name, and each item in attraction_genres
df_tm['genre_bucket'] = df_tm['genre_name'].apply(map_to_bucket)
df_tm['attraction_genre_buckets'] = df_tm['attraction_genres'].apply(
    lambda genres: list({map_to_bucket(g) for g in genres}) if isinstance(genres, list) else []
)

# Skiddle — apply to each item in genres
df_sk['genre_buckets'] = df_sk['genres'].apply(
    lambda genres: list({map_to_bucket(g) for g in genres}) if isinstance(genres, list) else []
)

In [84]:
unmapped_tm = df_tm[df_tm['genre_bucket'] == 'Other/Unmapped']['genre_name'].unique()
print("Unmapped Ticketmaster genres:", unmapped_tm)

Unmapped Ticketmaster genres: []


In [85]:
sk_all_used = set()
for g_list in df_sk['genres'].dropna():
    if isinstance(g_list, list):
        sk_all_used.update(g_list)

unmapped_sk = [g for g in sk_all_used if map_to_bucket(g) == 'Other/Unmapped']
print("Unmapped Skiddle genres:", unmapped_sk)

Unmapped Skiddle genres: []


##### Step 5: Cleanining region names

In [175]:
# Clean Ticketmaster's city_name — strip the comma-suffix
df_tm['city_clean'] = df_tm['city_name'].str.split(',').str[0].str.strip().str.title()

In [176]:
# verifiying above works
pprint(df_tm['city_clean'].unique().tolist())

['Narberth',
 'Glasgow',
 'London',
 'Hornchurch',
 'Cardiff',
 'Newcastle',
 'Llangollen',
 'Margate',
 'Manchester',
 'York',
 'Tynemouth',
 'Scarborough',
 'Halifax',
 'Staffordshire',
 'Birmingham',
 'Brighton',
 'Leeds',
 'Edinburgh',
 'Aberdeen',
 'High Wycombe',
 'Truro',
 'Derby',
 'Newcastle Upon Tyne',
 'Southampton',
 'Bath',
 'Peterborough',
 'Colchester',
 'Hertford',
 'Leamington Spa',
 'Norwich',
 'Walton-On-Thames',
 'Stevenage',
 'Stockport',
 'St Austell',
 'Nottingham',
 'Swansea',
 'Liverpool',
 'Luton',
 'Nuneaton',
 'Bristol',
 'Watford',
 'Torquay',
 'Dunfermline',
 'Salisbury',
 'Belfast',
 'Llandudno',
 'Sheffield',
 'Cumbria',
 'Dover',
 'Wolverhampton',
 'Inverness',
 'Portsmouth',
 'St Albans',
 'Tunbridge Wells',
 'Newbury',
 'Maidstone',
 'Newmarket',
 'Folkestone',
 'Wrexham',
 'Coventry',
 'Stockton-On-Tees',
 'Cambridge',
 'Newton-Le-Willows',
 'Perth',
 'Reading',
 'Bathgate',
 'Whitby',
 'Salford',
 'Exeter',
 'South Shields',
 'Saltaire',
 'Southend-

In [173]:
# Clean Skiddle's venue.town — exclude non-UK entries, normalize case
non_uk_towns = ['Paris']  

df_sk = df_sk[~df_sk['venue.town'].isin(non_uk_towns)].copy()
df_sk['city_clean'] = df_sk['venue.town'].str.strip().str.title()

In [177]:
# verifiying above works
pprint(df_sk['city_clean'].unique().tolist())

['Grantham',
 'London',
 'Weston Super-Mare',
 'Huntingdon',
 'Liverpool',
 'Leeds',
 'Maldon',
 'Cheltenham',
 'Manchester',
 'Nottingham',
 'Southampton',
 'Carlisle',
 'Shildon',
 'Lewes',
 'Letchworth',
 'Beckenham',
 'Hampstead',
 'Perranporth',
 'Rotherham',
 'Plymouth',
 'Kettering',
 'Reading',
 'Birmingham',
 'Sheffield',
 'Clifton',
 'Southend-On-Sea',
 'Basingstoke',
 'Newcastle',
 'Preston',
 'Leicester',
 'Doncaster',
 'Margate, Kent',
 'Angus',
 'Norwich',
 'Stockton-On-Tees',
 'Camberley',
 'Burnley',
 'Milton Keynes',
 'Glasgow',
 'Bristol',
 'Winchester',
 'Billericay',
 'Batley',
 'Greenwich',
 'Trowbridge',
 'Salisbury',
 'Derby',
 'Leicestershire',
 'Surrey',
 'Malmesbury',
 'Tetbury',
 'Holyhead',
 'Lockerbie',
 'Nr Cleobury Mortimer',
 'Stockport',
 'Witney',
 'Brighton',
 'Tamworth',
 'Holmfirth',
 'Hove',
 'Wimbledon',
 'Belfast',
 'Salford',
 'Cleethorpes',
 'Chadderton',
 'Leigh',
 'Chesterfield',
 'Cambridgeshire',
 'Newcastle Upon Tyne',
 'Portsmouth',
 'Hud

In [179]:
# verifiying above works
mask = df_sk['venue.region'] == 'France'
df_venue = df_sk[['city_clean','venue.region']]

france_region = df_venue[mask]
france_region

,city_clean,venue.region


##### Step 5: Combining the tables

In [89]:
df_sk['duration_days'] = (pd.to_datetime(df_sk['enddate']) - pd.to_datetime(df_sk['startdate'])).dt.days

In [181]:
df_tm_final = df_tm[['id', 'name', 'dates.start.localDate', 'city_clean', 'genres_bucketed', 'attraction_count']].copy()
df_tm_final = df_tm_final.rename(columns={'dates.start.localDate': 'date', 'city_clean': 'city'})
df_tm_final['source'] = 'ticketmaster'

df_sk_final = df_sk[['id', 'eventname', 'startdate', 'city_clean', 'genre_buckets', 'artist_count', 'duration_days']].copy()
df_sk_final = df_sk_final.rename(columns={'eventname': 'name', 'startdate': 'date', 'city_clean': 'city', 'genre_buckets': 'genres_bucketed'})
df_sk_final['source'] = 'skiddle'

df_combined = pd.concat([df_tm_final, df_sk_final], ignore_index=True)
df_combined.shape

(3101, 9)

In [ ]:
df_sk['duration_days'] = (pd.to_datetime(df_sk['enddate']) - pd.to_datetime(df_sk['startdate'])).dt.days

In [187]:
df_combined.head()

,id,name,date,city,genres_bucketed,source,duration_days
0,LvZ18QxAj1bZeL8vGSGnc,EVERYWHERE AT ONCE: Dewin,2026-06-28,Narberth,[Rock/Indie],ticketmaster,NaN
1,LvZ18QO6E5_0DZYvqZ7DL,King Tuts Summer Nights - Golden Ticket,2026-07-09,Glasgow,[Rock/Indie],ticketmaster,NaN
2,16djZbzkpG7X9uN,Friday Day - Wireless 2026,2026-07-10,London,[Unclassified],ticketmaster,NaN
3,1AMZkGyGkdEauCg,Amex Presents BST Hyde Park - Pitbull - Ultima...,2026-07-10,London,[Unclassified],ticketmaster,NaN
4,1AMZkGyGkeJTygI,American Express Presents BST Hyde Park - Pitbull,2026-07-10,London,[Unclassified],ticketmaster,NaN


##### Checking nulls

In [93]:
def null_vals(dataframe):
    """
    Show both number of nulls and the percentage of nulls in the whole column across a Pandas dataframe.
    """
    null_vals = dataframe.isnull().sum() # How many nulls in each column
    total_cnt = len(dataframe) # Total entries in the dataframe
    null_vals = pd.DataFrame(null_vals,columns=['null']) # Put the number of nulls in a single dataframe
    null_vals['percent'] = round((null_vals['null']/total_cnt)*100,3) # Round how many nulls are there, as %, of the df
    
    return null_vals.sort_values('percent', ascending=False) # Ordered from MOST to LEAST nulls

In [190]:
null_vals(df_combined)

,null,percent
city,1,0.032
id,0,0.000
name,0,0.000
date,0,0.000
genres_bucketed,0,0.000
source,0,0.000


##### Dropping artist_count, attraction_count & duration_days

In [189]:
df_combined = df_combined.drop(columns=['attraction_count', 'artist_count', 'duration_days'], errors='ignore')
# duration_days — your call, keep if you want scale-by-genre as a secondary finding

##### Decided to keep duration_days even with high nulls - I will filter the streamlit app based on whether data source is ticketmaster or skiddle. So duration_days will be relevant for skiddle filter.

In [191]:
df_combined.to_csv("df_combined_genre.csv", index=False)
print(f"Final combined dataset: {len(df_combined)} rows")

Final combined dataset: 3101 rows
